# 🛠️ Utilidades — Normalización de nombres de jugadores

Herramientas auxiliares para mejorar la precisión en la identificación de jugadores.
No forman parte del pipeline principal pero complementan el sistema.

## 📋 Utilidad 1 — Scraping de jugadores de LaLiga (Transfermarkt)

Obtiene la lista oficial de jugadores de LaLiga desde Transfermarkt y la guarda en `jugadores_laliga.json`.
Se ejecuta una vez por temporada.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import os
import json

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
TEMPORADA = 2025

def obtener_equipos_laliga() -> list:
    """Obtiene la lista de equipos de LaLiga con sus URLs de plantilla."""
    url = "https://www.transfermarkt.es/laliga/startseite/wettbewerb/ES1"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    equipos = []
    vistos = set()

    for a in soup.select("table.items tbody tr td.hauptlink a"):
        nombre = a.text.strip()
        href   = a["href"]

        if not nombre or nombre in vistos or "startseite" not in href:
            continue

        # Construir URL de plantilla: startseite → kader
        url_kader = "https://www.transfermarkt.es" + href.replace("startseite", "kader")
        equipos.append({"nombre": nombre, "url_kader": url_kader})
        vistos.add(nombre)

    return equipos


def obtener_jugadores_equipo(equipo: dict) -> list:
    """Obtiene la plantilla completa de un equipo desde Transfermarkt."""
    response = requests.get(equipo["url_kader"], headers=headers)
    soup     = BeautifulSoup(response.content, "html.parser")

    jugadores = []
    vistos    = set()

    for fila in soup.select("table.items tbody tr"):
        nombre_tag   = fila.select_one("td.hauptlink a")
        # La posición está en el segundo <tr> de la inline-table
        posicion_tag = fila.select_one("td.posrela table tr:nth-of-type(2) td")

        if not nombre_tag:
            continue

        nombre = nombre_tag.text.strip()
        if not nombre or nombre in vistos:
            continue

        jugadores.append({
            "nombre":   nombre,
            "posicion": posicion_tag.text.strip() if posicion_tag else None,
            "equipo":   equipo["nombre"]
        })
        vistos.add(nombre)

    return jugadores


def scraping_jugadores_laliga() -> list:
    """Scraping completo de todos los jugadores de LaLiga."""
    print("🔍 Obteniendo equipos de LaLiga...\n")
    equipos = obtener_equipos_laliga()
    print(f"✅ {len(equipos)} equipos encontrados\n")

    todos_jugadores = []

    for i, equipo in enumerate(equipos, 1):
        print(f"  [{i}/{len(equipos)}] {equipo['nombre']}...")
        jugadores = obtener_jugadores_equipo(equipo)
        todos_jugadores.extend(jugadores)
        print(f"           ✅ {len(jugadores)} jugadores")
        time.sleep(1)  # Respetar rate limit de Transfermarkt

    return todos_jugadores


# --- Ejecutar scraping ---
if os.path.exists("jugadores_laliga.json"):
    print("📂 Cargando jugadores desde JSON local...")
    with open("jugadores_laliga.json", "r", encoding="utf-8") as f:
        jugadores_laliga = json.load(f)
    print(f"✅ {len(jugadores_laliga)} jugadores cargados")

else:
    print("⚠️  JSON no encontrado. Ejecutando scraping...\n")
    jugadores_laliga = scraping_jugadores_laliga()

    with open("jugadores_laliga.json", "w", encoding="utf-8") as f:
        json.dump(jugadores_laliga, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Total jugadores: {len(jugadores_laliga)}")
    print(f"💾 Guardados en: jugadores_laliga.json")

# Vista previa
print(f"\n👀 Primeros 5 jugadores:")
for j in jugadores_laliga[:5]:
    print(f"  {j['nombre']} — {j['equipo']} — {j['posicion']}")

# Fuzzy matching

In [ ]:
!pip install rapidfuzz -q

In [ ]:
from rapidfuzz import process, fuzz

# Cargar jugadores oficiales
with open("jugadores_laliga.json", "r", encoding="utf-8") as f:
    jugadores_laliga = json.load(f)

# Lista de nombres oficiales para comparar
nombres_oficiales = [j["nombre"] for j in jugadores_laliga]

# Nombres problemáticos detectados en nuestros insights
nombres_test = [
    "Rafiña",
    "Chamenamení",
    "Yugrinich",
    "William Cruzeta",
    "Gerard de Santico Mesaña",
    "Robert Navarro",
    "la mine",
]

print("🔍 Test de fuzzy matching:\n")
for nombre in nombres_test:
    resultado = process.extractOne(
        nombre,
        nombres_oficiales,
        scorer=fuzz.WRatio,  # Combina varios algoritmos, más robusto
        score_cutoff=50      # Descarta coincidencias por debajo del 50%
    )
    if resultado:
        print(f"  '{nombre}' → '{resultado[0]}' (similitud: {resultado[1]:.0f}%)")
    else:
        print(f"  '{nombre}' → ❌ Sin coincidencia")

In [ ]:
!pip install openai -q
from openai import OpenAI
from getpass import getpass

client_test = OpenAI(api_key=getpass("🔑 OpenAI API Key: "))

In [ ]:
# Test normalización de nombres dudosos con OpenAI

nombres_dudosos_test = [
    "Rafiña",
    "Chamenamení",
    "Yugrinich",
    "William Cruzeta",
    "Gerard de Santico Mesaña",
    "la mine",
]

prompt = f"""Tienes una lista de nombres de jugadores mal transcritos automáticamente por YouTube.
Normaliza cada nombre al nombre oficial más probable de la lista de jugadores de LaLiga.
Si no encuentras coincidencia clara, devuelve null.

NOMBRES A NORMALIZAR:
{json.dumps(nombres_dudosos_test, ensure_ascii=False)}

LISTA OFICIAL DE JUGADORES:
{json.dumps(nombres_oficiales, ensure_ascii=False)}

Devuelve ÚNICAMENTE un JSON sin markdown:
{{"normalizaciones": [{{"original": "nombre_mal_transcrito", "oficial": "nombre_oficial_o_null"}}]}}"""

response = client_test.chat.completions.create(
    model       = "gpt-4o-mini",
    max_tokens  = 500,
    temperature = 0,
    messages    = [{"role": "user", "content": prompt}]
)

resultado = json.loads(response.choices[0].message.content)
print("🔍 Resultados normalización:\n")
for r in resultado["normalizaciones"]:
    print(f"  '{r['original']}' → '{r['oficial']}'")

In [ ]:
# Buscar manualmente en la lista oficial
busquedas = ["Tchouameni", "Tchouaméni", "Camavinga", "Bellingham", "Vinicius", "Lamine"]

for b in busquedas:
    encontrados = [j["nombre"] for j in jugadores_laliga if b.lower() in j["nombre"].lower()]
    print(f"  '{b}' → {encontrados}")

In [ ]:
prompt = f"""Eres un experto en fútbol español. Tienes nombres de jugadores mal transcritos
automáticamente por YouTube — pueden estar distorsionados fonéticamente, en español incorrecto,
o ser apodos.

Tu tarea es identificar a qué jugador oficial de LaLiga corresponde cada nombre distorsionado,
buscando similitud fonética, no solo ortográfica.

Ejemplos de distorsiones típicas:
- "la mine" → "Lamine Yamal" (apodo fonético)
- "Rafiña" → "Raphinha" (pronunciación española)
- "Chamenamení" → "Aurélien Tchouaméni" (distorsión fonética fuerte)

NOMBRES A NORMALIZAR:
{json.dumps(nombres_dudosos_test, ensure_ascii=False)}

LISTA OFICIAL DE JUGADORES DE LALIGA:
{json.dumps(nombres_oficiales, ensure_ascii=False)}

Devuelve ÚNICAMENTE un JSON sin markdown:
{{"normalizaciones": [{{"original": "nombre_mal_transcrito", "oficial": "nombre_oficial_o_null"}}]}}"""

response = client_test.chat.completions.create(
    model       = "gpt-4o-mini",
    max_tokens  = 500,
    temperature = 0,
    messages    = [{"role": "user", "content": prompt}]
)

resultado = json.loads(response.choices[0].message.content)
print("🔍 Resultados normalización:\n")
for r in resultado["normalizaciones"]:
    print(f"  '{r['original']}' → '{r['oficial']}'")

In [ ]:
!pip install jellyfish -q

In [ ]:
import jellyfish
from rapidfuzz import process, fuzz

nombres_dudosos_test = [
    "Witchel",
    "Setber",
    "Aram barry",
    "Rafiña",
    "Chamenamení",
    "Yugrinich",
    "William Cruzeta",
    "Gerard de Santico Mesaña",
    "la mine",
    "Guruzeta",
    "Santi Comesaña",
    "Lejend"
]

def top5_rapidfuzz(nombre: str, nombres_oficiales: list) -> list:
    resultados = process.extract(
        nombre,
        nombres_oficiales,
        scorer   = fuzz.WRatio,
        limit    = 5,
        score_cutoff = 40
    )
    return [{"nombre": r[0], "score": round(r[1], 1)} for r in resultados]

def top5_jellyfish(nombre: str, nombres_oficiales: list) -> list:
    scores = []
    for oficial in nombres_oficiales:
        score = jellyfish.jaro_winkler_similarity(nombre.lower(), oficial.lower())
        scores.append({"nombre": oficial, "score": round(score * 100, 1)})
    scores.sort(key=lambda x: x["score"], reverse=True)
    return scores[:5]

print("🔍 Test rapidfuzz + jellyfish:\n")
for nombre in nombres_dudosos_test:
    print(f"'{nombre}'")
    print(f"  rapidfuzz : {top5_rapidfuzz(nombre, nombres_oficiales)}")
    print(f"  jellyfish : {top5_jellyfish(nombre, nombres_oficiales)}")
    print()

In [ ]:
from rapidfuzz import process, fuzz

scorers = {
    "WRatio":            fuzz.WRatio,
    "token_sort_ratio":  fuzz.token_sort_ratio,
    "token_set_ratio":   fuzz.token_set_ratio,
    "partial_ratio":     fuzz.partial_ratio,
    "ratio":             fuzz.ratio,
}

nombres_dudosos_test = [
    "Witchel",
    "Setber",
    "Aram barry",
    "Rafiña",
    "Chumenamení",
    "Yugrinich",
    "William Cruzeta",
    "Gerard de Santico Mesaña",
    "la mine",
    "Hanco",
    "Feng Gruver",
    "Iga Chaniga"
]

nombre_test = "Aram barry"

print(f"🔍 Comparativa de scorers para '{nombre_test}':\n")
for nombre_scorer, scorer in scorers.items():
    resultados = process.extract(
        nombre_test,
        nombres_oficiales,
        scorer       = scorer,
        limit        = 5,
        score_cutoff = 30
    )
    print(f"  {nombre_scorer}:")
    for r in resultados:
        print(f"    {r[0]} ({r[1]:.1f}%)")
    print()

In [ ]:
def normalizar_con_openai(nombre: str, nombres_oficiales: list) -> str:
    """
    Combina rapidfuzz (múltiples scorers) + jellyfish y pasa todos los candidatos
    a OpenAI para que tome la decisión final.
    """
    # Obtener candidatos de cada scorer
    candidatos = {}
    for nombre_scorer, scorer in scorers.items():
        resultados = process.extract(
            nombre,
            nombres_oficiales,
            scorer       = scorer,
            limit        = 5,
            score_cutoff = 30
        )
        candidatos[nombre_scorer] = [{"nombre": r[0], "score": round(r[1], 1)} for r in resultados]

    # Candidatos jellyfish
    candidatos["jellyfish_jaro_winkler"] = top5_jellyfish(nombre, nombres_oficiales)

    prompt = f"""Eres un experto en fútbol español. Tu tarea es identificar a qué jugador oficial
de LaLiga corresponde un nombre mal transcrito automáticamente por YouTube.

NOMBRE MAL TRANSCRITO: "{nombre}"

CANDIDATOS SEGÚN DIFERENTES ALGORITMOS:
{json.dumps(candidatos, ensure_ascii=False, indent=2)}

Cada algoritmo mide similitud de forma diferente:
- WRatio: similitud general combinada
- token_sort_ratio: similitud ignorando orden de palabras
- token_set_ratio: similitud por intersección de palabras
- partial_ratio: similitud parcial (subcadenas)
- ratio: similitud carácter a carácter
- jellyfish_jaro_winkler: similitud fonética

Analiza todos los candidatos y sus scores para decidir el nombre oficial más probable.
Ten en cuenta que la transcripción automática de YouTube distorsiona fonéticamente los nombres extranjeros.
Si ningún candidato es convincente, devuelve null.

Devuelve ÚNICAMENTE un JSON sin markdown:
{{"oficial": "nombre_oficial_o_null", "razon": "explicacion_breve"}}"""

    response = client_test.chat.completions.create(
        model       = "gpt-4o-mini",
        max_tokens  = 200,
        temperature = 0,
        messages    = [{"role": "user", "content": prompt}]
    )

    return json.loads(response.choices[0].message.content)


# --- TEST ---
print("🔍 Test normalización combinada:\n")
for nombre in nombres_dudosos_test:
    resultado = normalizar_con_openai(nombre, nombres_oficiales)
    print(f"  '{nombre}'")
    print(f"  → {resultado['oficial']}")
    print(f"  → {resultado['razon']}")
    print()

In [ ]:
scorers = {
    "WRatio":            fuzz.WRatio,
    "token_sort_ratio":  fuzz.token_sort_ratio,
    "token_set_ratio":   fuzz.token_set_ratio,
    "partial_ratio":     fuzz.partial_ratio,
    "ratio":             fuzz.ratio,
}

nombres_dudosos_test = [
    "Witchel",
    "Setber",
    "Arambarry",
    "Rafiña",
    "Chumenamení",
    "Yugrinich",
    "William Cruzeta",
    "Gerard de Santico Mesaña",
    "Aran Barry",
    "Hanco",
    "Legend",
    "Iga Chaniga"
]

def normalizar_con_openai(nombre: str, nombres_oficiales: list) -> dict:
    """
    Obtiene candidatos con rapidfuzz (múltiples scorers) y pasa la lista a OpenAI
    para que tome la decisión final.
    """
    # Obtener candidatos de cada scorer sin scores
    candidatos = {}
    for nombre_scorer, scorer in scorers.items():
        resultados = process.extract(
            nombre,
            nombres_oficiales,
            scorer       = scorer,
            limit        = 5,
            score_cutoff = 30
        )
        candidatos[nombre_scorer] = [r[0] for r in resultados]

    prompt = f"""Eres un experto en fútbol español. Un sistema de transcripción automática
de YouTube ha generado el siguiente nombre de jugador, que puede estar distorsionado
fonéticamente, mal pronunciado o ser un apodo en español.

NOMBRE TRANSCRITO POR YOUTUBE: "{nombre}"

CANDIDATOS SUGERIDOS POR DIFERENTES ALGORITMOS DE SIMILITUD:
{json.dumps(candidatos, ensure_ascii=False, indent=2)}

Teniendo en cuenta que el nombre viene de una transcripción automática de YouTube
y puede estar fonéticamente distorsionado, escoge el jugador oficial de LaLiga
que creas más acertado. Si ningún candidato es convincente, devuelve null.

Devuelve ÚNICAMENTE un JSON sin markdown:
{{"oficial": "nombre_oficial_o_null"}}"""

    response = client_test.chat.completions.create(
        model       = "gpt-4o-mini",
        max_tokens  = 200,
        temperature = 0,
        messages    = [{"role": "user", "content": prompt}]
    )

    return json.loads(response.choices[0].message.content)


# --- TEST ---
print("🔍 Test normalización combinada:\n")
for nombre in nombres_dudosos_test:
    resultado = normalizar_con_openai(nombre, nombres_oficiales)
    print(f"  '{nombre}' → '{resultado['oficial']}'")
    print()

# 🏆 Fantasy Copilot — Agente Analista con IA

> Pipeline de extracción y análisis automático de contenido de YouTube para recomendaciones de alineación en LaLiga Fantasy, basado en RAG y modelos de lenguaje.

## ⚙️ Bloque 1 — Instalación de librerías

Instalamos todas las dependencias necesarias para el proyecto.

| Librería | Uso |
|---|---|
| `youtube-transcript-api` | Descarga de transcripciones de YouTube |
| `openai` | Extracción de insights y embeddings |
| `chromadb` | Base de datos vectorial para el RAG |
| `google-api-python-client` | YouTube Data API v3 |
| `gradio` | Interfaz de chatbot |
| `tiktoken` | Cálculo de tokens y costes |

In [ ]:
!pip install youtube-transcript-api==1.2.4 -q
!pip install openai -q
!pip install chromadb -q
!pip install google-api-python-client -q
!pip install gradio -q
!pip install tiktoken -q

print("✅ Librerías instaladas")

## 🔑 Bloque 2 — Imports y configuración

Importamos todas las librerías necesarias e inicializamos los clientes de OpenAI y YouTube.

Las API keys se introducen en runtime mediante `getpass`. Nunca quedan expuestas en el código.

In [ ]:
import os
import json
import re
from getpass import getpass
from datetime import datetime
from typing import Optional

from openai import OpenAI
import tiktoken
from youtube_transcript_api import YouTubeTranscriptApi
from googleapiclient.discovery import build
import gradio as gr

# --- API KEYS ---
OPENAI_API_KEY  = getpass("🔑 Introduce tu OpenAI API Key: ")
YOUTUBE_API_KEY = getpass("🔑 Introduce tu YouTube API Key: ")

# Inicializar cliente OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

# Verificar conexión
test = client.chat.completions.create(
    model="gpt-5.4-nano",
    max_completion_tokens=20,
    messages=[{"role": "user", "content": "Di Hola en español de España"}]
)
print("✅ OpenAI conectado:", test.choices[0].message.content)
print("✅ Imports completados")

## 📺 Bloque 3 — Obtención de videos de YouTube

Conectamos con la YouTube Data API para obtener los videos de análisis fantasy de los canales registrados.

**Decisiones de diseño:**
- Los channel IDs se hardcodean en `CANALES_FANTASY` para evitar llamadas innecesarias a la API
- Cada canal tiene sus propias `keywords` para filtrar videos relevantes
- Se excluyen videos del Mundial y videos sin número de jornada identificable
- Los videos se filtran por jornada objetivo (`JORNADA_OBJETIVO = 38`)
- Los resultados se ordenan por fecha descendente. El video más reciente de cada jornada tiene más peso

**Canales registrados:**
- 📺 Carrasco
- 📺 Jornada Perfecta Fantasy

> Para añadir un nuevo canal en el futuro, basta con añadir una entrada al diccionario `CANALES_FANTASY` con su `channel_id` y `keywords`.

In [ ]:
youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

CANALES_FANTASY = {
    "carrasco": {
        "channel_id": "UCu7ZOzCUgWpvvJzYAdnw0_Q",
        "keywords": ["jornada"]
    },
    "jornada_perfecta": {
        "channel_id": "UCvMw41zGXyYYPDvyXcHZfug",
        "keywords": ["jornada", " j3", " j4"]  # Cubre "J38", "J37", etc. con espacio previo para evitar falsos positivos
    },
}

def obtener_uploads_playlist_id(channel_id: str) -> str:
    response = youtube.channels().list(
        id=channel_id,
        part="contentDetails"
    ).execute()
    return response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

def es_video_relevante(titulo: str, keywords: list) -> bool:
    titulo_lower = titulo.lower()

    # Filtros de exclusión
    if "mundial" in titulo_lower:
        return False

    # Debe contener alguna keyword
    if not any(kw.lower() in titulo_lower for kw in keywords):
        return False

    # Debe tener número de jornada identificable (elimina los J?)
    if extraer_numero_jornada(titulo) is None:
        return False

    return True

def extraer_numero_jornada(titulo: str) -> Optional[int]:
    """Extrae el número de jornada del título. Soporta 'Jornada 38' y 'J38'."""
    match = re.search(r'(?:jornada\s*|j)(\d+)', titulo.lower())
    return int(match.group(1)) if match else None

def obtener_videos_canal(canal_key: str, max_videos: int = 50, solo_relevantes: bool = True, jornada_filtro: Optional[int] = None) -> list:
    if canal_key not in CANALES_FANTASY:
        raise ValueError(f"Canal '{canal_key}' no registrado. Disponibles: {list(CANALES_FANTASY.keys())}")

    canal_info = CANALES_FANTASY[canal_key]
    channel_id = canal_info["channel_id"]
    keywords   = canal_info["keywords"]

    playlist_id = obtener_uploads_playlist_id(channel_id)
    response = youtube.playlistItems().list(
        playlistId=playlist_id,
        part="id,snippet",
        maxResults=max_videos
    ).execute()

    videos = []
    for item in response.get("items", []):
        titulo   = item["snippet"]["title"]
        video_id = item["snippet"]["resourceId"]["videoId"]

        if solo_relevantes and not es_video_relevante(titulo, keywords):
            continue

        jornada = extraer_numero_jornada(titulo)

        # Filtrar por jornada específica si se indica
        if jornada_filtro is not None and jornada != jornada_filtro:
            continue

        videos.append({
            "canal":    canal_key,
            "video_id": video_id,
            "titulo":   titulo,
            "fecha":    item["snippet"]["publishedAt"][:10],
            "jornada":  jornada,
            "url":      f"https://youtube.com/watch?v={video_id}"
        })

    videos.sort(key=lambda v: (v["jornada"] or 0, v["fecha"]), reverse=True)
    return videos


# --- TEST ambos canales, solo J38 ---
JORNADA_OBJETIVO = 38

for canal_key in CANALES_FANTASY.keys():
    print(f"\n📹 Videos J{JORNADA_OBJETIVO} de [{canal_key}]:\n")
    videos = obtener_videos_canal(canal_key, max_videos=50, solo_relevantes=True, jornada_filtro=JORNADA_OBJETIVO)

    if not videos:
        print(f"  ⚠️  Sin videos de J{JORNADA_OBJETIVO}.")
    else:
        for v in videos:
            jornada_str = f"J{v['jornada']}" if v['jornada'] else "J?"
            print(f"  [{jornada_str}] [{v['fecha']}] {v['titulo']}")

## 📝 Bloque 4 — Extracción de transcripciones

Descargamos las transcripciones en crudo de cada video usando `youtube-transcript-api`.

**Problema conocido:** YouTube bloquea peticiones desde IPs de servidores cloud como Google Colab. Se soluciona usando un proxy residencial de [Webshare](https://www.webshare.io).

**Flujo:**
1. Si `transcripciones_j38.json` existe en disco → carga directamente sin llamar a YouTube
2. Si no existe → descarga todas las transcripciones a través del proxy y guarda el JSON

> 💾 **Archivo generado:** `transcripciones_j38.json`. Descárgalo y consérvalo entre sesiones para evitar repetir las llamadas a YouTube.

**Credenciales necesarias:**
- Proxy Username y Proxy Password de Webshare (plan gratuito es suficiente)

In [ ]:
import os

if os.path.exists("transcripciones_j38.json"):
    print("📂 Cargando transcripciones desde JSON local...")
    with open("transcripciones_j38.json", "r", encoding="utf-8") as f:
        videos_con_transcripcion = json.load(f)
    ok    = sum(1 for v in videos_con_transcripcion if v["estado"] == "ok")
    error = sum(1 for v in videos_con_transcripcion if v["estado"] != "ok")
    print(f"✅ {len(videos_con_transcripcion)} videos cargados — {ok} ✅ con transcripción — {error} ❌ sin transcripción")

else:
    print("⚠️  JSON no encontrado.")

In [ ]:
from youtube_transcript_api.proxies import WebshareProxyConfig

WEBSHARE_USERNAME = getpass("🔑 Introduce tu Webshare Proxy Username: ")
WEBSHARE_PASSWORD = getpass("🔑 Introduce tu Webshare Proxy Password: ")

In [ ]:
# Test directo de credenciales Webshare
from youtube_transcript_api.proxies import WebshareProxyConfig

ytt_test = YouTubeTranscriptApi(
    proxy_config=WebshareProxyConfig(
        proxy_username=WEBSHARE_USERNAME,
        proxy_password=WEBSHARE_PASSWORD,
    )
)

try:
    fragmentos = ytt_test.fetch("AT_UTIeNmtI", languages=["es"])
    texto = " ".join(f.text for f in fragmentos)
    print(f"✅ Funciona: {len(texto):,} caracteres")
    print(texto[:200])
except Exception as e:
    print(f"❌ Error completo: {type(e).__name__}")
    print(str(e))

In [ ]:
ytt_api = YouTubeTranscriptApi(
    proxy_config=WebshareProxyConfig(
        proxy_username=WEBSHARE_USERNAME,
        proxy_password=WEBSHARE_PASSWORD,
    )
)

def extraer_transcripcion(video: dict) -> dict:
    """Descarga la transcripción en crudo de un video."""
    video_id = video["video_id"]
    try:
        fragmentos  = ytt_api.fetch(video_id, languages=["es", "en"])
        texto_crudo = " ".join(fragmento.text for fragmento in fragmentos)
        return {**video, "transcripcion_cruda": texto_crudo, "estado": "ok"}
    except Exception as e:
        return {**video, "transcripcion_cruda": None, "estado": f"error: {type(e).__name__}: {str(e)[:80]}"}


def extraer_todas_transcripciones(videos: list) -> list:
    """Itera todos los videos y extrae sus transcripciones."""
    resultados = []
    total      = len(videos)

    for i, video in enumerate(videos, 1):
        print(f"  [{i}/{total}] {video['canal']} — {video['titulo'][:60]}...")
        resultado = extraer_transcripcion(video)

        if resultado["estado"] == "ok":
            chars = len(resultado["transcripcion_cruda"])
            print(f"           ✅ {chars:,} caracteres")
        else:
            print(f"           ❌ {resultado['estado']}")

        resultados.append(resultado)

    return resultados


# --- Obtener todos los videos ---
todos_videos = []
for canal_key in CANALES_FANTASY.keys():
    videos = obtener_videos_canal(canal_key, max_videos=50, solo_relevantes=True, jornada_filtro=JORNADA_OBJETIVO)
    todos_videos.extend(videos)

print(f"📹 Total videos a procesar: {len(todos_videos)}\n")

# --- Test previo con un solo video ---
print("🧪 Test con primer video...\n")
test = extraer_transcripcion(todos_videos[0])
if test["estado"] == "ok":
    print(f"✅ Proxy funcionando — {len(test['transcripcion_cruda']):,} caracteres")
    print(f"Primeros 200 chars: {test['transcripcion_cruda'][:200]}\n")

    # Si el test va bien, extraer todos
    print("🚀 Extrayendo todas las transcripciones...\n")
    videos_con_transcripcion = extraer_todas_transcripciones(todos_videos)

    # Guardar en JSON local
    with open("transcripciones_j38.json", "w", encoding="utf-8") as f:
        json.dump(videos_con_transcripcion, f, ensure_ascii=False, indent=2)

    ok    = sum(1 for v in videos_con_transcripcion if v["estado"] == "ok")
    error = sum(1 for v in videos_con_transcripcion if v["estado"] != "ok")
    print(f"\n📊 Resumen: {ok} ✅ con transcripción — {error} ❌ sin transcripción")
    print(f"💾 Guardadas en: transcripciones_j38.json")

else:
    print(f"❌ Proxy no funciona: {test['estado']}")
    print("Revisa tus credenciales de Webshare antes de continuar.")

In [ ]:
# Reintentar transcripciones fallidas

fallidos = [v for v in videos_con_transcripcion if v["estado"] != "ok"]
print(f"🔄 Reintentando {len(fallidos)} videos fallidos...\n")

for i, video in enumerate(fallidos, 1):
    print(f"  [{i}/{len(fallidos)}] {video['titulo'][:60]}...")
    resultado = extraer_transcripcion(video)

    if resultado["estado"] == "ok":
        chars = len(resultado["transcripcion_cruda"])
        print(f"           ✅ {chars:,} caracteres")
        # Actualizar en la lista principal
        for j, v in enumerate(videos_con_transcripcion):
            if v["video_id"] == video["video_id"]:
                videos_con_transcripcion[j] = resultado
                break
    else:
        print(f"           ❌ {resultado['estado']}")

# Guardar JSON actualizado
with open("transcripciones_j38.json", "w", encoding="utf-8") as f:
    json.dump(videos_con_transcripcion, f, ensure_ascii=False, indent=2)

ok    = sum(1 for v in videos_con_transcripcion if v["estado"] == "ok")
error = sum(1 for v in videos_con_transcripcion if v["estado"] != "ok")
print(f"\n📊 Resumen actualizado: {ok} ✅ — {error} ❌")
print(f"💾 JSON actualizado: transcripciones_j38.json")

## 🧹 Bloque 5 — Limpieza y control de calidad

Procesamos las transcripciones en crudo para dejarlas listas para enviar a OpenAI.

**Pasos:**
1. **Limpieza**: eliminamos etiquetas `[Música]`, `[Aplausos]`, caracteres extraños y espacios múltiples
2. **Control de calidad**: clasificamos cada transcripción según su longitud:
   - `< 500 chars` → descartada (transcripción rota o vacía)
   - `500 - 30.000 chars` → se procesa directamente
   - `> 30.000 chars` → se divide en chunks
3. **Chunking** — dividimos los textos largos en fragmentos de `10.000 chars` con `500 chars` de solapamiento para no perder contexto entre chunks.

**Configuración ajustable:**

| Parámetro | Valor | Descripción |
|---|---|---|
| `MIN_CHARS` | 500 | Mínimo para no descartar |
| `MAX_CHARS` | 30.000 | Máximo antes de chunkear |
| `CHUNK_SIZE` | 10.000 | Tamaño de cada chunk |
| `CHUNK_OVERLAP` | 500 | Solapamiento entre chunks |

> El output de este bloque es una lista plana de textos listos para OpenAI, donde cada elemento conoce su canal, video y jornada de origen.

In [ ]:
# --- Configuración ---
MIN_CHARS     = 500
MAX_CHARS     = 15_000
CHUNK_SIZE    = 10_000
CHUNK_OVERLAP = 500

def limpiar_texto(texto: str) -> str:
    """Limpia artefactos típicos de transcripciones automáticas de YouTube."""
    # Eliminar etiquetas tipo [Música], [Aplausos], [Risas]
    texto = re.sub(r'\[.*?\]', '', texto)
    # Eliminar caracteres especiales extraños manteniendo puntuación española
    texto = re.sub(r'[^\w\s\.,;:¿?¡!áéíóúüñÁÉÍÓÚÜÑ]', ' ', texto)
    # Colapsar espacios múltiples y saltos de línea
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()


def chunkear_texto(texto: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    chunks = []
    start  = 0

    while start < len(texto):
        end = start + chunk_size

        if end >= len(texto):
            chunks.append(texto[start:].strip())
            break

        # Buscar corte limpio al final del chunk
        corte = end
        for separador in ['. ', '? ', '! ', ', ', '; ']:
            pos = texto.rfind(separador, end - 200, end)
            if pos != -1 and pos > corte - 200:
                corte = pos + len(separador)
                break

        chunks.append(texto[start:corte].strip())

        # Buscar inicio limpio del siguiente chunk (aplicar overlap desde corte)
        inicio_overlap = corte - overlap
        for separador in ['. ', '? ', '! ', ', ', '; ']:
            pos = texto.find(separador, inicio_overlap, inicio_overlap + 100)
            if pos != -1:
                start = pos + len(separador)
                break
        else:
            start = inicio_overlap

    return [c for c in chunks if c]


def procesar_calidad(video: dict) -> list:
    """
    Limpia la transcripción y aplica control de calidad.
    Retorna lista de dicts listos para OpenAI:
    - 1 elemento si el texto cabe entero
    - N elementos si necesita chunkeo
    - 0 elementos si es descartado
    """
    # Sin transcripción
    if video["estado"] != "ok" or not video["transcripcion_cruda"]:
        return []

    texto_limpio = limpiar_texto(video["transcripcion_cruda"])
    chars        = len(texto_limpio)

    # Demasiado corta
    if chars < MIN_CHARS:
        print(f"  ⚠️  Descartado (muy corta: {chars} chars): {video['titulo'][:50]}")
        return []

    base = {
        "canal":   video["canal"],
        "titulo":  video["titulo"],
        "fecha":   video["fecha"],
        "jornada": video["jornada"],
        "url":     video["url"],
    }

    # Longitud correcta: procesar entero
    if chars <= MAX_CHARS:
        return [{**base, "texto": texto_limpio, "chunk": None, "total_chunks": 1}]

    # Demasiado larga: chunkear
    chunks = chunkear_texto(texto_limpio)
    print(f"  ✂️  Chunkeado en {len(chunks)} partes: {video['titulo'][:50]}")
    return [
        {**base, "texto": chunk, "chunk": i + 1, "total_chunks": len(chunks)}
        for i, chunk in enumerate(chunks)
    ]


def preparar_textos_para_llm(videos_con_transcripcion: list) -> list:
    """Aplica limpieza y control de calidad a todos los videos."""
    textos_listos = []
    total = len(videos_con_transcripcion)

    for i, video in enumerate(videos_con_transcripcion, 1):
        titulo_corto = video['titulo'][:55]
        resultados   = procesar_calidad(video)

        if not resultados:
            if video["estado"] == "ok":
                pass  # Ya imprime el warning dentro de procesar_calidad
            else:
                print(f"  [{i}/{total}] ❌ Sin transcripción: {titulo_corto}")
            continue

        chars = len(resultados[0]["texto"])
        chunks_str = f"{resultados[0]['total_chunks']} chunk(s)"
        print(f"  [{i}/{total}] ✅ {chars:,} chars — {chunks_str} — {titulo_corto}")
        textos_listos.extend(resultados)

    return textos_listos


# --- TEST ---
print("🧹 Limpiando y clasificando transcripciones...\n")
textos_para_llm = preparar_textos_para_llm(videos_con_transcripcion)

print(f"\n📊 Resumen:")
print(f"  Videos procesados : {len(videos_con_transcripcion)}")
print(f"  Textos listos     : {len(textos_para_llm)}")
print(f"\nEjemplo — primeros 300 chars del primer texto:")
print(textos_para_llm[0]["texto"][:300] if textos_para_llm else "Sin datos")

In [ ]:
# Vista previa de lo que recibirá el Bloque 6
print("👀 Vista previa de chunks para OpenAI:\n")
for i, t in enumerate(textos_para_llm, 1):
    chunk_str = f"chunk {t['chunk']}/{t['total_chunks']}" if t['total_chunks'] > 1 else "completo"
    print(f"  [{i}] {t['canal']} | {t['fecha']} | J{t['jornada']} | {chunk_str}")
    print(f"       Título : {t['titulo'][:60]}")
    print(f"       Chars  : {len(t['texto']):,}")
    print(f"       Texto  : {t['texto'][:150]}...")
    print()

## 🤖 Bloque 6 — Extracción de insights con OpenAI

Procesamos cada chunk de transcripción con OpenAI para extraer insights estructurados sobre jugadores.

**Flujo:**
1. Si `insights_j38.json` existe en disco → carga directamente sin llamar a OpenAI
2. Si no existe → procesa todos los chunks y guarda el JSON

**Para cada chunk, OpenAI extrae:**
- Jugador y posición
- Acción recomendada — `comprar`, `vender`, `alinear` o `evitar`
- Confianza (0.0 - 1.0) — basada en el tono del analista
- Razón corta y criterios de análisis
- Quote original del analista si existe
- Flag `nombre_dudoso`: indica si el nombre puede estar mal transcrito

**Decisiones técnicas:**
- Modelo `gpt-5.4-nano`: buena relación calidad/coste para extracción estructurada
- `temperature=0.1`: respuestas consistentes y predecibles
- **Structured outputs**: el formato JSON se garantiza mediante un schema (`json_schema`), no por instrucciones en el prompt. Elimina por completo los errores de parseo
- Diccionario `ALIASES_JUGADORES` como respaldo para apodos conocidos

> 💾 **Archivo generado:** `insights_j38.json`. Descárgalo y consérvalo entre sesiones. Es el dato más importante del proyecto.

**Resultado:** ~577 insights estructurados extraídos de 17 videos de la Jornada 38.

In [ ]:
# En caso de no tener los insights generados

# --- Schema para structured outputs ---
INSIGHT_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "insights_extraidos",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "insights": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "jugador":        {"type": "string"},
                            "accion":         {"type": "string", "enum": ["alinear", "evitar"]},
                            "confianza":      {"type": "number"},
                            "razon_corta":    {"type": "string"},
                            "criterios": {
                                "type": "object",
                                "properties": {
                                    "forma_actual":      {"type": ["string", "null"]},
                                    "dinamica_equipo":   {"type": ["string", "null"]},
                                    "calendario_rival":  {"type": ["string", "null"]},
                                    "prediccion_puntos": {"type": ["string", "null"]}
                                },
                                "required": ["forma_actual", "dinamica_equipo", "calendario_rival", "prediccion_puntos"],
                                "additionalProperties": False
                            },
                            "quote_original": {"type": ["string", "null"]}
                        },
                        "required": ["jugador", "accion", "confianza", "razon_corta", "criterios", "quote_original"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["insights"],
            "additionalProperties": False
        }
    }
}

PROMPT_EXTRACCION = """Eres un experto analista de fantasy football español.
Se te proporciona la transcripción de un video de YouTube de un canal de análisis fantasy de LaLiga.

CONTEXTO DEL VIDEO:
- Canal: {canal}
- Título: {titulo}
- Fecha: {fecha}
- Jornada: {jornada}

TRANSCRIPCIÓN:
{texto}

INSTRUCCIONES:
1. Extrae TODOS los insights sobre jugadores específicos que contengan una recomendación clara.
2. Escribe el nombre del jugador EXACTAMENTE como aparece o se menciona en la transcripción — no intentes corregirlo ni normalizarlo.
3. Incluye jugadores que interpretes que tienen recomendación explícita en el texto: alinear o evitar.
4. La confianza (0.0-1.0) debe reflejar cuán seguro suena el analista.
5. Si el analista duda, baja la confianza. Si es tajante, súbela.
6. Si un criterio no se menciona, usa null.
7. Si no hay insights claros devuelve lista vacía. En este caso, antes de devolver lista vacía, revisa bien y no devuelvas lista vacía a no ser que estés totalmente seguro de que no hay insights."""


In [ ]:
# Encoder de gpt-4o
enc = tiktoken.encoding_for_model("gpt-4o-mini")

# Calcular tokens del prompt base (sin transcripción)
prompt_base = PROMPT_EXTRACCION.format(
    canal="carrasco",
    titulo="Test",
    fecha="2026-05-23",
    jornada=38,
    texto=""
)
tokens_prompt_base = len(enc.encode(prompt_base))

# Calcular tokens de cada chunk
total_tokens_input  = 0
total_tokens_output = 0  # Estimado: ~500 tokens de output por chunk

print(f"📊 Análisis de tokens por chunk:\n")
print(f"  {'#':<4} {'Canal':<20} {'Chars':>8} {'Tokens texto':>14} {'Tokens total':>13}")
print(f"  {'-'*65}")

for i, chunk in enumerate(textos_para_llm, 1):
    tokens_texto = len(enc.encode(chunk["texto"]))
    tokens_total = tokens_prompt_base + tokens_texto
    total_tokens_input += tokens_total
    print(f"  {i:<4} {chunk['canal']:<20} {len(chunk['texto']):>8,} {tokens_texto:>14,} {tokens_total:>13,}")

total_tokens_output = len(textos_para_llm) * 500  # Estimado output

# Precios gpt-4o-mini (junio 2026)
precio_input  = (total_tokens_input  / 1_000_000) * 0.20
precio_output = (total_tokens_output / 1_000_000) * 1.25
precio_total  = precio_input + precio_output

print(f"\n📊 Resumen:")
print(f"  Chunks totales       : {len(textos_para_llm)}")
print(f"  Tokens prompt base   : {tokens_prompt_base:,}")
print(f"  Tokens input total   : {total_tokens_input:,}")
print(f"  Tokens output est.   : {total_tokens_output:,}")
print(f"\n💰 Coste estimado:")
print(f"  Input  : ${precio_input:.4f}")
print(f"  Output : ${precio_output:.4f}")
print(f"  TOTAL  : ${precio_total:.4f}")

In [ ]:
import time

def extraer_insights_openai(texto_chunk: dict, max_reintentos: int = 3) -> list:
    for intento in range(1, max_reintentos + 1):
        try:
            prompt = PROMPT_EXTRACCION.format(
                canal=texto_chunk["canal"],
                titulo=texto_chunk["titulo"],
                fecha=texto_chunk["fecha"],
                jornada=texto_chunk["jornada"],
                texto=texto_chunk["texto"]
            )

            response = client.chat.completions.create(
                model           = "gpt-5.4-nano",
                max_completion_tokens = 6000,
                temperature     = 0.1,
                messages        = [{"role": "user", "content": prompt}],
                response_format = INSIGHT_SCHEMA
            )

            contenido = response.choices[0].message.content

            if not contenido or not contenido.strip():
                print(f"    ⚠️  Intento {intento}/{max_reintentos} — Respuesta vacía")
                time.sleep(1)
                continue

            data     = json.loads(contenido)
            insights = data["insights"]

            for insight in insights:
                insight["fuente"] = {
                    "canal":   texto_chunk["canal"],
                    "titulo":  texto_chunk["titulo"],
                    "fecha":   texto_chunk["fecha"],
                    "jornada": texto_chunk["jornada"],
                    "url":     texto_chunk["url"],
                    "chunk":   texto_chunk["chunk"]
                }
            return insights

        except json.JSONDecodeError as e:
            print(f"    ⚠️  Intento {intento}/{max_reintentos} — Error JSON: {str(e)[:80]}")
            time.sleep(1)
        except Exception as e:
            print(f"    ⚠️  Intento {intento}/{max_reintentos} — Error: {type(e).__name__}: {str(e)[:80]}")
            time.sleep(1)

    print(f"    ❌ Descartado tras {max_reintentos} intentos")
    return []

def construir_base_conocimiento(textos_para_llm: list) -> list:
    """Itera todos los chunks y extrae insights con OpenAI."""
    todos_insights = []
    total = len(textos_para_llm)

    for i, chunk in enumerate(textos_para_llm, 1):
        chunk_str = f"chunk {chunk['chunk']}/{chunk['total_chunks']}" if chunk['total_chunks'] > 1 else "completo"
        print(f"  [{i}/{total}] {chunk['canal']} | J{chunk['jornada']} | {chunk_str}")
        print(f"           {chunk['titulo'][:60]}")

        insights = extraer_insights_openai(chunk)
        todos_insights.extend(insights)
        print(f"           ✅ {len(insights)} insights extraídos")

        time.sleep(0.5)  # Pequeña pausa entre llamadas para evitar rate limits

    return todos_insights

In [ ]:
# TEST con solo el segundo chunk
print("🤖 Test con segundo chunk...\n")

chunk_test = textos_para_llm[5]
print(f"📹 Video : {chunk_test['titulo'][:70]}")
print(f"📦 Chunk : {chunk_test['chunk']}/{chunk_test['total_chunks']}")
print(f"📝 Chars : {len(chunk_test['texto']):,}\n")
print(f"⬆️ Texto : {chunk_test['texto']}\n")

insights_test = extraer_insights_openai(chunk_test)

print(f"📊 Insights extraídos: {len(insights_test)}\n")
for insight in insights_test:
    dudoso_str = " ⚠️ NOMBRE DUDOSO" if insight.get("nombre_dudoso") else ""
    print(f"  {insight['jugador']} → {insight['accion'].upper()} | confianza: {insight['confianza']}")
    print(f"  Razón: {insight['razon_corta']}")
    print()

In [ ]:
# --- Cargar desde JSON o regenerar ---
if os.path.exists("insights_j38.json"):
    print("📂 Cargando insights desde JSON local...")
    with open("insights_j38.json", "r", encoding="utf-8") as f:
        todos_insights = json.load(f)
    print(f"✅ {len(todos_insights)} insights cargados desde insights_j38.json")

else:
    print("⚠️  JSON no encontrado. Extrayendo insights con OpenAI...\n")
    todos_insights = construir_base_conocimiento(textos_para_llm)

    if todos_insights:
        with open("insights_j38.json", "w", encoding="utf-8") as f:
            json.dump(todos_insights, f, ensure_ascii=False, indent=2)
        print(f"\n📊 Total insights extraídos: {len(todos_insights)}")
        print(f"💾 Guardados en: insights_j38.json")
    else:
        print("\n⚠️  No se extrajeron insights — revisa los errores anteriores")

In [ ]:
# Relanzar chunks fallidos (0 insights extraídos)
chunks_fallidos = []

for i, chunk in enumerate(textos_para_llm):
    insights_chunk = [
        ins for ins in todos_insights
        if ins["fuente"]["url"]   == chunk["url"]
        and ins["fuente"]["chunk"] == chunk["chunk"]
    ]
    if len(insights_chunk) == 0:
        chunks_fallidos.append((i, chunk))

print(f"🔄 Chunks sin insights detectados: {len(chunks_fallidos)}\n")

In [ ]:
nuevos_insights = []
for i, chunk in chunks_fallidos:
    chunk_str = f"chunk {chunk['chunk']}/{chunk['total_chunks']}" if chunk['chunk'] else "completo"
    print(f"  [{i+1}/{len(textos_para_llm)}] {chunk['canal']} | {chunk_str}")
    print(f"           {chunk['titulo'][:60]}")

    insights = extraer_insights_openai(chunk)
    nuevos_insights.extend(insights)
    print(f"           ✅ {len(insights)} insights extraídos")

# Añadir nuevos insights a los existentes
todos_insights.extend(nuevos_insights)

# Guardar JSON actualizado
with open("insights_j38.json", "w", encoding="utf-8") as f:
    json.dump(todos_insights, f, ensure_ascii=False, indent=2)

print(f"\n📊 Insights previos    : {len(todos_insights) - len(nuevos_insights)}")
print(f"📊 Insights nuevos     : {len(nuevos_insights)}")
print(f"📊 Total insights final: {len(todos_insights)}")
print(f"💾 JSON actualizado    : insights_j38.json")

### Normalización de nombres

In [ ]:
with open("insights_j38.json", "r", encoding="utf-8") as f:
    todos_insights = json.load(f)

nombres_unicos = list(set([i["jugador"] for i in todos_insights]))
print(f"Total insights: {len(todos_insights)}")
print(f"Nombres únicos: {len(nombres_unicos)}")
print(f"\nEjemplos de nombres:")
for n in sorted(nombres_unicos)[:20]:
    print(f"  {n}")

In [ ]:
from rapidfuzz import process, fuzz
import time

def obtener_candidatos(nombre: str, nombres_oficiales: list, top_k: int = 5) -> dict:
    """Obtiene candidatos únicos por scorer sin scores, solo nombres."""
    scorers = {
        "WRatio":           fuzz.WRatio,
        "token_sort_ratio": fuzz.token_sort_ratio,
        "token_set_ratio":  fuzz.token_set_ratio,
        "partial_ratio":    fuzz.partial_ratio,
        "ratio":            fuzz.ratio,
    }

    candidatos_por_scorer = {}
    for nombre_scorer, scorer in scorers.items():
        resultados = process.extract(
            nombre,
            nombres_oficiales,
            scorer       = scorer,
            limit        = top_k,
            score_cutoff = 30
        )
        candidatos_por_scorer[nombre_scorer] = [r[0] for r in resultados]

    return candidatos_por_scorer

In [ ]:
# Definición de tools para normalización
TOOLS_NORMALIZACION = [
    {
        "type": "function",
        "function": {
            "name": "buscar_en_lista_oficial",
            "description": "Busca si un nombre existe exactamente o casi exactamente en la lista oficial de jugadores de LaLiga. Úsala siempre como primer paso.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nombre": {
                        "type": "string",
                        "description": "Nombre a buscar en la lista oficial"
                    }
                },
                "required": ["nombre"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_por_equipo",
            "description": "Filtra jugadores de la lista oficial por equipo. Úsala cuando en la quote aparezca mencionado un equipo de LaLiga.",
            "parameters": {
                "type": "object",
                "properties": {
                    "equipo": {
                        "type": "string",
                        "description": "Nombre del equipo mencionado en la quote"
                    }
                },
                "required": ["equipo"]
            }
        }
    }
]

In [ ]:
NORMALIZACION_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "normalizacion_jugador",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "nombre_oficial": {"type": ["string", "null"]}
            },
            "required": ["nombre_oficial"],
            "additionalProperties": False
        }
    }
}

In [ ]:
PROMPT_NORMALIZACION_V2 = """Eres un experto lingüista y aficionado al fútbol español.

Una transcripción automática de YouTube ha generado un nombre de jugador que puede estar:
- Fonéticamente distorsionado (ej: "mastantono" → "Franco Mastantuono")
- En español coloquial (ej: "Rafiña" → "Raphinha")
- Ser solo el apellido o parte de él (ej: "feng gruver" → "David Affengruber" porque "gruver" está contenido en "Affengruber")
- Cortado o fragmentado por la transcripción

NOMBRE TRANSCRITO: "{nombre}"
QUOTE ORIGINAL: "{quote}"

CANDIDATOS POR ALGORITMO DE SIMILITUD:
{candidatos}

HERRAMIENTAS DISPONIBLES:
1. buscar_en_lista_oficial — úsala cuando no estés seguro de si el nombre existe tal cual
2. buscar_por_equipo — úsala cuando en la quote aparezca mencionado un equipo de LaLiga

PROCESO:
1. Analiza el nombre transcrito — ¿tiene forma de nombre o apellido de persona?
2. Analiza los candidatos — ¿alguno contiene fonéticamente o textualmente el nombre transcrito?
3. Analiza la quote — ¿confirma o contradice algún candidato?
4. Si tienes dudas → usa las herramientas, MÁXIMO 3 llamadas en total

CRITERIOS DE RECHAZO OBLIGATORIO — devuelve null si:
- El nombre transcrito COMPLETO no tiene forma de nombre o apellido de persona —
  si contiene palabras que claramente no forman parte de un nombre propio, rechaza aunque
  alguna palabra suelta coincida con un jugador
- No hay conexión clara entre el nombre transcrito completo y algún candidato —
  una coincidencia parcial de una sola palabra no es suficiente
- La quote no tiene coherencia con una recomendación sobre un jugador de fútbol real
- Tienes cualquier duda razonable

Es mejor eliminar que normalizar incorrectamente.
Analiza siempre el nombre transcrito en su totalidad, nunca por partes aisladas."""

In [ ]:
def ejecutar_tool_normalizacion(nombre_tool: str, argumentos: dict) -> str:
    """Ejecuta la tool solicitada por el modelo y retorna resultado como string."""

    if nombre_tool == "buscar_en_lista_oficial":
        nombre_buscar = argumentos["nombre"]
        # Búsqueda exacta
        if nombre_buscar in jugadores_por_nombre:
            j = jugadores_por_nombre[nombre_buscar]
            return f"Encontrado exactamente: {j['nombre']} ({j['equipo']}, {j['posicion']})"
        # Búsqueda fuzzy estricta (>90%)
        resultado = process.extractOne(
            nombre_buscar,
            nombres_oficiales,
            scorer       = fuzz.WRatio,
            score_cutoff = 90
        )
        if resultado:
            j = jugadores_por_nombre[resultado[0]]
            return f"Encontrado con alta similitud: {j['nombre']} ({j['equipo']}, {j['posicion']})"
        return f"No encontrado con alta similitud para: {nombre_buscar}"

    elif nombre_tool == "buscar_por_equipo":
        equipo_buscar = argumentos["equipo"]
        # Buscar equipo con fuzzy matching
        equipos_unicos = list(set([j["equipo"] for j in jugadores_laliga]))
        equipo_oficial = process.extractOne(
            equipo_buscar,
            equipos_unicos,
            scorer       = fuzz.WRatio,
            score_cutoff = 60
        )
        if not equipo_oficial:
            return f"Equipo no encontrado: {equipo_buscar}"

        jugadores_equipo = [j["nombre"] for j in jugadores_laliga if j["equipo"] == equipo_oficial[0]]
        return f"Jugadores de {equipo_oficial[0]}: {json.dumps(jugadores_equipo, ensure_ascii=False)}"

    return "Tool no reconocida"

def normalizar_nombre_v2(nombre: str, quote: str, nombres_oficiales: list) -> dict:
    candidatos = obtener_candidatos(nombre, nombres_oficiales)
    prompt = PROMPT_NORMALIZACION_V2.format(
        nombre     = nombre,
        quote      = quote or "No disponible",
        candidatos = json.dumps(candidatos, ensure_ascii=False, indent=2)
    )

    mensajes = [
        {"role": "user",   "content": prompt}
    ]

    # Primera fase — function calling
    while True:
        response = client.chat.completions.create(
            model       = "gpt-5.4-nano",
            max_completion_tokens  = 500,
            temperature = 0,
            messages    = mensajes,
            tools       = TOOLS_NORMALIZACION,
            tool_choice = "auto"
        )

        mensaje = response.choices[0].message

        if mensaje.tool_calls:
            mensajes.append(mensaje)
            for tool_call in mensaje.tool_calls:
                nombre_tool = tool_call.function.name
                argumentos  = json.loads(tool_call.function.arguments)
                resultado   = ejecutar_tool_normalizacion(nombre_tool, argumentos)
                print(f"    🔧 {nombre_tool}({argumentos}) → {resultado[:80]}")
                mensajes.append({
                    "role":         "tool",
                    "tool_call_id": tool_call.id,
                    "content":      resultado
                })
        else:
            # Añadir respuesta final al historial
            mensajes.append({"role": "assistant", "content": mensaje.content})
            break

    # Segunda fase — structured output
    mensajes.append({
        "role":    "user",
        "content": "Basándote en toda la investigación anterior, devuelve el nombre oficial del jugador."
    })

    response_final = client.chat.completions.create(
        model           = "gpt-5.4-nano",
        max_completion_tokens = 50,
        temperature     = 0,
        messages        = mensajes,
        response_format = NORMALIZACION_SCHEMA
    )

    return json.loads(response_final.choices[0].message.content)

In [ ]:
# Test de normalización con nombres problemáticos
nombres_test = [
    ("Catena", None),
    ("mastantono", '"En el Real Madrid vamos a poner… mastan tuono"'),
    ("Rafiña", '"Rafiña es una compra clarísima"'),
    ("feng gruver", '"feng gruver juega en el Elche"'),
    ("Acor Camavinga", '"Acor Camavinga juega de titular"'),
    ("75 kg para Moreno", None),
    ("Abcar del Getafe", None),
    ("Pepelu", "...en el Valencia también recibimos un cambio respecto a lo de ayer y es que jugaré finalmente Pepelu como central, Unay Núñez en el lateral derecho..."),
    ("Legend", "Yo tengo a Legend en la liga de mis amigos puesto como titular"),
    ("Aram Barry", "Cuando un Getafe tiene que ganar, el gol pasa por un hombre, el uruguayo, Arambarry, posiblemente último partido en el Colise porque puede marcharse a Argentina")
]

# Siempre reconstruir al inicio de la Fase 2
with open("jugadores_laliga.json", "r", encoding="utf-8") as f:
    jugadores_laliga = json.load(f)

nombres_oficiales    = [j["nombre"] for j in jugadores_laliga]
jugadores_por_nombre = {j["nombre"]: j for j in jugadores_laliga}

# --- TEST ---
print("🔍 Test normalización con function calling:\n")
for nombre, quote in nombres_test:
    print(f"\n  Nombre: '{nombre}'")
    resultado      = normalizar_nombre_v2(nombre, quote or "", nombres_oficiales)
    nombre_oficial = resultado["nombre_oficial"]

    if nombre_oficial and nombre_oficial in jugadores_por_nombre:
        jugador_info = jugadores_por_nombre[nombre_oficial]
        print(f"  ✅ → '{nombre_oficial}' ({jugador_info.get('equipo', '?')})")
    else:
        print(f"  ❌ → eliminado")

    time.sleep(0.5)

In [ ]:
def ejecutar_fase2(todos_insights: list, jugadores_laliga: list) -> list:
    """
    Fase 2 completa:
    1. Normaliza nombres únicos con rapidfuzz + OpenAI (function calling)
    2. Completa equipo y posición desde jugadores_laliga.json
    3. Actualiza insights y elimina los no normalizados
    """
    # Índice de jugadores oficiales para búsqueda rápida
    nombres_oficiales    = [j["nombre"] for j in jugadores_laliga]
    jugadores_por_nombre = {j["nombre"]: j for j in jugadores_laliga}

    # Extraer nombres únicos con su quote más representativo
    nombres_con_quote = {}
    for insight in todos_insights:
        nombre = insight["jugador"]
        if nombre not in nombres_con_quote:
            nombres_con_quote[nombre] = insight.get("quote_original") or ""

    total = len(nombres_con_quote)
    print(f"🔍 Normalizando {total} nombres únicos...\n")

    # Normalizar cada nombre único
    mapa_normalizacion = {}
    eliminados         = []
    total_procesados   = 0

    for nombre, quote in nombres_con_quote.items():
        total_procesados += 1
        resultado      = normalizar_nombre_v2(nombre, quote, nombres_oficiales)
        nombre_oficial = resultado["nombre_oficial"]

        if nombre_oficial and nombre_oficial in jugadores_por_nombre:
            mapa_normalizacion[nombre] = nombre_oficial
            jugador_info = jugadores_por_nombre[nombre_oficial]
            print(f"  [{total_procesados}/{total}] ✅ '{nombre}' → '{nombre_oficial}' ({jugador_info['equipo']})")
        else:
            mapa_normalizacion[nombre] = None
            eliminados.append({"nombre": nombre, "quote": quote})
            print(f"  [{total_procesados}/{total}] ❌ '{nombre}' → eliminado")

        time.sleep(0.5)

    # Actualizar insights con nombre oficial, equipo y posición
    print(f"\n📝 Actualizando insights...")
    insights_normalizados = []
    insights_eliminados   = 0

    for insight in todos_insights:
        nombre_original = insight["jugador"]
        nombre_oficial  = mapa_normalizacion.get(nombre_original)

        if nombre_oficial and nombre_oficial in jugadores_por_nombre:
            jugador_info       = jugadores_por_nombre[nombre_oficial]
            insight["jugador"]  = nombre_oficial
            insight["equipo"]   = jugador_info["equipo"]
            insight["posicion"] = jugador_info["posicion"]
            insights_normalizados.append(insight)
        else:
            insights_eliminados += 1

    print(f"\n📊 Resumen Fase 2:")
    print(f"  Nombres únicos procesados : {total}")
    print(f"  Nombres normalizados      : {len(mapa_normalizacion) - len(eliminados)}")
    print(f"  Nombres eliminados        : {len(eliminados)}")
    print(f"  Insights finales          : {len(insights_normalizados)}")
    print(f"  Insights eliminados       : {insights_eliminados}")

    # Guardar log de eliminados
    with open("nombres_eliminados.json", "w", encoding="utf-8") as f:
        json.dump(eliminados, f, ensure_ascii=False, indent=2)
    print(f"  Log eliminados            : nombres_eliminados.json")

    return insights_normalizados

In [ ]:
# --- EJECUTAR FASE 2 ---
with open("insights_j38.json", "r", encoding="utf-8") as f:
    todos_insights = json.load(f)

with open("jugadores_laliga.json", "r", encoding="utf-8") as f:
    jugadores_laliga = json.load(f)

insights_normalizados = ejecutar_fase2(todos_insights, jugadores_laliga)

# Guardar insights normalizados
with open("insights_j38_normalizado.json", "w", encoding="utf-8") as f:
    json.dump(insights_normalizados, f, ensure_ascii=False, indent=2)

print(f"\n💾 Guardado: insights_j38_normalizado.json")

## 🔍 Bloque 7 — RAG con ChromaDB

Indexamos los insights en una base de datos vectorial para poder hacer búsquedas semánticas.

**¿Por qué RAG?**
El RAG (Retrieval Augmented Generation) nos permite buscar los insights más relevantes para una pregunta sin tener que pasar todos los datos al modelo — más eficiente, más preciso y más económico.

**Flujo:**
1. Cada insight se convierte en un texto indexable con toda su información semántica
2. OpenAI genera un embedding (`text-embedding-3-small`) por cada texto
3. Los embeddings se almacenan en ChromaDB con similitud coseno
4. Ante una consulta, se buscan los `top_k` insights más similares semánticamente

**Decisiones técnicas:**
- Modelo `text-embedding-3-small`: mejor calidad que SentenceTransformers en español, coste muy bajo (~$0.01 para 577 insights)
- Similitud coseno: más adecuada que distancia euclidiana para comparar textos
- `TOP_K = 10`: recuperamos hasta 10 insights por consulta
- `filtrar_dudosos`: parámetro opcional para excluir nombres dudosos del retrieval

**Nota:** ChromaDB se inicializa en memoria: se pierde al cerrar la sesión. Al reiniciar, basta con ejecutar este bloque para reindexar desde `insights_j38.json` en ~30 segundos.

**Mejora futura:** complementar el RAG con una base de datos de grafo (Neo4j) para modelar relaciones entre jugadores, equipos y jornadas. Especialmente útil para consultas complejas como *"jugadores del Real Madrid con rival fácil esta jornada"*.

In [ ]:
import chromadb

# --- Configuración ---
MODELO_EMBEDDINGS = "text-embedding-3-small"
TOP_K             = 10

def generar_embedding(texto: str) -> list:
    """Genera embedding de un texto usando OpenAI."""
    response = client.embeddings.create(
        model = MODELO_EMBEDDINGS,
        input = texto
    )
    return response.data[0].embedding


def generar_embeddings_batch(textos: list, batch_size: int = 100) -> list:
    """
    Genera embeddings en batches para no saturar la API.
    OpenAI permite hasta 2048 inputs por llamada pero usamos 100 por seguridad.
    """
    embeddings = []
    total      = len(textos)

    for i in range(0, total, batch_size):
        batch = textos[i:i + batch_size]
        print(f"  Batch {i//batch_size + 1}/{(total//batch_size) + 1} — {len(batch)} textos...")
        response = client.embeddings.create(
            model = MODELO_EMBEDDINGS,
            input = batch
        )
        embeddings.extend([e.embedding for e in response.data])

    return embeddings


# Inicializar ChromaDB
chroma_client = chromadb.Client()
coleccion     = chroma_client.get_or_create_collection(
    name     = "fantasy_insights",
    metadata = {"hnsw:space": "cosine"}
)
print("✅ ChromaDB inicializado")


def insight_a_texto(insight: dict) -> str:
    """Convierte un insight a texto indexable para el embedding."""
    criterios = insight.get("criterios", {})
    partes = [
        f"{insight['jugador']}, {insight.get('posicion', '')}.",
        f"Recomendación: {insight['accion']}.",
        f"Razón: {insight.get('razon_corta', '')}.",
        f"Forma actual: {criterios.get('forma_actual', '')}." if criterios.get('forma_actual') else "",
        f"Dinámica equipo: {criterios.get('dinamica_equipo', '')}." if criterios.get('dinamica_equipo') else "",
        f"Calendario rival: {criterios.get('calendario_rival', '')}." if criterios.get('calendario_rival') else "",
        f"Predicción puntos: {criterios.get('prediccion_puntos', '')}." if criterios.get('prediccion_puntos') else "",
        f"Fuente: {insight['fuente']['canal']}, Jornada {insight['fuente']['jornada']}.",
    ]
    return " ".join([p for p in partes if p])


def indexar_insights(insights: list) -> None:
    """Genera embeddings e indexa todos los insights en ChromaDB."""
    if not insights:
        print("⚠️  No hay insights para indexar.")
        return

    textos    = [insight_a_texto(i) for i in insights]
    ids       = [f"insight_{idx}" for idx in range(len(insights))]
    metadatas = [
        {
            "jugador":       i["jugador"],
            "posicion":      i.get("posicion", ""),
            "accion":        i["accion"],
            "confianza":     float(i["confianza"]),
            "canal":         i["fuente"]["canal"],
            "fecha":         i["fuente"]["fecha"],
            "jornada":       int(i["fuente"]["jornada"]) if i["fuente"]["jornada"] else 0,
            "nombre_dudoso": bool(i.get("nombre_dudoso", False)),
            "url":           i["fuente"]["url"],
        }
        for i in insights
    ]

    print(f"⏳ Generando embeddings para {len(textos)} insights...")
    embeddings = generar_embeddings_batch(textos)

    coleccion.add(
        ids        = ids,
        documents  = textos,
        embeddings = embeddings,
        metadatas  = metadatas
    )
    print(f"✅ {len(insights)} insights indexados en ChromaDB")


def consultar_rag(pregunta: str, top_k: int = TOP_K, filtrar_dudosos: bool = False) -> list:
    """Busca los insights más relevantes para una pregunta."""
    embedding_pregunta = generar_embedding(pregunta)

    where = {"nombre_dudoso": False} if filtrar_dudosos else None

    resultados = coleccion.query(
        query_embeddings = [embedding_pregunta],
        n_results        = top_k,
        where            = where
    )

    insights_relevantes = []
    for i in range(len(resultados["ids"][0])):
        insights_relevantes.append({
            "texto":     resultados["documents"][0][i],
            "metadata":  resultados["metadatas"][0][i],
            "distancia": resultados["distances"][0][i],
        })

    return insights_relevantes


# --- TEST ---
print(f"\n📦 Indexando {len(todos_insights)} insights...\n")
indexar_insights(todos_insights)

print("\n🔍 Test de consulta RAG:\n")
pregunta_test = "¿Debo alinear a Pedri esta jornada?"
resultados    = consultar_rag(pregunta_test, top_k=3)

print(f"Pregunta: '{pregunta_test}'")
print(f"RAG encontró {len(resultados)} documentos relevantes:\n")
for i, r in enumerate(resultados, 1):
    m = r["metadata"]
    print(f"  [{i}] {m['jugador']} → {m['accion'].upper()} | confianza: {m['confianza']} | {m['canal']}")
    print(f"       {r['texto'][:120]}...")
    print(f"       Similitud: {1 - r['distancia']:.2f}")
    print()

## ⚽ Bloque 8 — Agente Analista

Genera una recomendación de alineación óptima para la jornada basándose en los insights del RAG.

**Flujo en dos fases:**

**Fase 1 — Resumen individual por jugador:**
- Para cada jugador de la plantilla, consulta el RAG con `top_k=10`
- Filtra insights con confianza `> 0.4` para eliminar ruido
- Llamada individual a OpenAI por jugador → resumen compacto y estructurado
- Resultado: un resumen por jugador con acción recomendada y confianza media

**Fase 2 — Recomendación final:**
- Todos los resúmenes individuales se pasan en una única llamada a OpenAI
- El modelo recomienda el 11 óptimo teniendo en cuenta posiciones y confianzas
- Resultado: alineación recomendada con justificación por jugador

**¿Por qué dos fases?**
Evitamos el problema de *"lost in the middle"*: los modelos pierden precisión cuando el contexto es muy largo. Cada llamada individual tiene contexto pequeño y enfocado, y la llamada final recibe resúmenes compactos en lugar de insights en bruto.

**Configuración:**

| Parámetro | Valor | Descripción |
|---|---|---|
| `UMBRAL_CONFIANZA` | 0.4 | Mínimo para considerar un insight |
| `TOP_K_RAG` | 10 | Insights máximos por jugador |
| `temperature` | 0.1 / 0.2 | Baja para respuestas consistentes |

> Introduce tu plantilla real en `MI_PLANTILLA` antes de ejecutar este bloque.

**Mejora futura:** añadir un agente especializado en optimización de alineación que resuelva el problema como programación lineal. Maximizar puntos esperados dado un presupuesto y restricciones de posición.

In [ ]:
# --- Configuración ---
UMBRAL_CONFIANZA = 0.4
TOP_K_RAG        = 10
JORNADA_OBJETIVO = 38

MI_PLANTILLA = [
    {"nombre": "Augusto Batalla",     "posicion": "portero"},
    {"nombre": "Abel Bretones",         "posicion": "defensa"},
    {"nombre": "Andrei Ratiu",    "posicion": "defensa"},
    {"nombre": "Carlos Romero",         "posicion": "defensa"},
    {"nombre": "David Affengruber", "posicion": "defensa"},
    {"nombre": "Javi Rodríguez", "posicion": "defensa"},
    {"nombre": "Léo Pétrot", "posicion": "defensa"},
    {"nombre": "Aleix Febas",          "posicion": "centrocampista"},
    {"nombre": "Chidera Ejuke",           "posicion": "centrocampista"},
    {"nombre": "Hugo Álvarez",          "posicion": "centrocampista"},
    {"nombre": "Ilaix Moriba",          "posicion": "centrocampista"},
    {"nombre": "Omar Mascarell",          "posicion": "centrocampista"},
    {"nombre": "Óscar Trejo",          "posicion": "centrocampista"},
    {"nombre": "Alemao",   "posicion": "delantero"},
    {"nombre": "Fede Viñas",          "posicion": "delantero"},
    {"nombre": "Gonzalo García",  "posicion": "delantero"},
    {"nombre": "Gonzalo García",  "posicion": "delantero"},
    {"nombre": "Goncalo Guedes",  "posicion": "delantero"},
    {"nombre": "Kike García",  "posicion": "delantero"},
    {"nombre": "Vedat Muriqi",  "posicion": "delantero"},
]

PROMPT_RESUMEN_JUGADOR = """Eres un experto analista de fantasy football español para LaLiga.

JUGADOR: {jugador} ({posicion})
JORNADA: {jornada}

INSIGHTS DE EXPERTOS SOBRE ESTE JUGADOR:
{insights}

Basándote SOLO en estos insights, genera un resumen conciso del jugador para esta jornada.

Devuelve ÚNICAMENTE un JSON válido sin markdown:
{{
  "jugador": "{jugador}",
  "posicion": "{posicion}",
  "confianza_media": 0.0,
  "accion_recomendada": "alinear|sentar|fichar|evitar",
  "resumen": "2-3 líneas máximo con lo más relevante",
  "fuentes": ["canal - título video"],
  "sin_datos": false
}}

Si no hay insights suficientes devuelve sin_datos: true y confianza_media: 0.0"""

PROMPT_RECOMENDACION_FINAL_PLANTILLA = """Eres un experto analista de fantasy football español para LaLiga.

JORNADA: {jornada}

RESÚMENES DE CADA JUGADOR DE LA PLANTILLA:
{resumenes}

Basándote en estos resúmenes, recomienda el 11 óptimo para esta jornada.
Ten en cuenta:
- Equilibrio por posiciones (1 portero, 4 defensas, 3-4 centrocampistas, 2-3 delanteros)
- Prioriza jugadores con mayor confianza
- Justifica los casos dudosos

Devuelve ÚNICAMENTE un JSON válido sin markdown:
{{
  "once_recomendado": [
    {{
      "jugador": "nombre",
      "posicion": "portero|defensa|centrocampista|delantero",
      "alinear": true,
      "confianza": 0.0,
      "justificacion": "razón breve"
    }}
  ],
  "advertencias": ["jugadores con dudas o sin datos"],
  "resumen": "resumen general en 2-3 líneas"
}}"""


def obtener_insights_jugador(nombre: str, top_k: int = TOP_K_RAG, umbral: float = UMBRAL_CONFIANZA) -> list:
    """
    Consulta el RAG y filtra por umbral de confianza.
    Retorna máximo top_k insights con confianza >= umbral.
    """
    pregunta   = f"¿Qué dicen los expertos sobre {nombre} en fantasy football?"
    resultados = consultar_rag(pregunta, top_k=top_k)

    # Filtrar por umbral de confianza
    filtrados = [r for r in resultados if r["metadata"]["confianza"] >= umbral]
    return filtrados


def resumir_jugador(jugador: dict, jornada: int = JORNADA_OBJETIVO) -> dict:
    """
    Llamada individual a OpenAI para resumir un jugador
    basándose en sus insights del RAG.
    """
    insights = obtener_insights_jugador(jugador["nombre"])

    # Formatear insights para el prompt
    if insights:
        insights_texto = "\n".join([
            f"- {r['metadata']['accion'].upper()} | confianza: {r['metadata']['confianza']} | {r['metadata']['canal']}\n  {r['texto'][:200]}"
            for r in insights
        ])
    else:
        insights_texto = "Sin insights disponibles para este jugador."

    prompt = PROMPT_RESUMEN_JUGADOR.format(
        jugador  = jugador["nombre"],
        posicion = jugador["posicion"],
        jornada  = jornada,
        insights = insights_texto
    )

    response = client.chat.completions.create(
        model       = "gpt-5.4-mini",
        max_completion_tokens  = 500,
        temperature = 0.1,
        messages    = [{"role": "user", "content": prompt}]
    )

    contenido = response.choices[0].message.content.strip()
    contenido = re.sub(r'^```json\s*', '', contenido)
    contenido = re.sub(r'\s*```$', '', contenido)

    try:
        resumen = json.loads(contenido)
        # Añadir precio si viene del mercado
        if "precio" in jugador:
            resumen["precio"] = jugador["precio"]
        print(f"  ✅ {jugador['nombre']} — {resumen.get('accion_recomendada', '?').upper()} | confianza: {resumen.get('confianza_media', 0)} | insights usados: {len(insights)}")
        return resumen
    except json.JSONDecodeError as e:
        print(f"  ⚠️  Error parseando JSON de {jugador['nombre']}: {e}")
        return {"jugador": jugador["nombre"], "posicion": jugador["posicion"], "sin_datos": True}


def analizar_plantilla(plantilla: list, jornada: int = JORNADA_OBJETIVO) -> dict:
    """
    Flujo completo de análisis de plantilla:
    1. Resumir cada jugador individualmente
    2. Llamada final con todos los resúmenes
    """
    print("🔍 Resumiendo jugadores de la plantilla...\n")
    resumenes = [resumir_jugador(j, jornada) for j in plantilla]

    print("\n🤖 Generando recomendación final de plantilla...")
    prompt = PROMPT_RECOMENDACION_FINAL_PLANTILLA.format(
        jornada  = jornada,
        resumenes = json.dumps(resumenes, ensure_ascii=False, indent=2)
    )

    response = client.chat.completions.create(
        model       = "gpt-5.4-mini",
        max_completion_tokens  = 2000,
        temperature = 0.2,
        messages    = [{"role": "user", "content": prompt}]
    )

    contenido = response.choices[0].message.content.strip()
    contenido = re.sub(r'^```json\s*', '', contenido)
    contenido = re.sub(r'\s*```$', '', contenido)

    try:
        return json.loads(contenido)
    except json.JSONDecodeError as e:
        print(f"⚠️  Error parseando JSON final: {e}")
        return {"error": contenido}

def mostrar_recomendacion_plantilla(resultado: dict) -> None:
    print("\n" + "="*60)
    print(f"🏆 ALINEACIÓN RECOMENDADA — J{JORNADA_OBJETIVO}")
    print("="*60)

    for jugador in resultado.get("once_recomendado", []):
        estado = "✅ ALINEAR" if jugador["alinear"] else "❌ SENTAR"
        print(f"\n  {estado} | {jugador['jugador']} ({jugador['posicion']})")
        print(f"  Confianza    : {jugador['confianza']}")
        print(f"  Justificación: {jugador['justificacion']}")

    if resultado.get("advertencias"):
        print("\n⚠️  ADVERTENCIAS:")
        for adv in resultado["advertencias"]:
            print(f"  - {adv}")

    print(f"\n📋 RESUMEN: {resultado.get('resumen', '')}")

# --- TEST ---
print("⚽ ANÁLISIS DE PLANTILLA\n")
resultado_plantilla = analizar_plantilla(MI_PLANTILLA)
mostrar_recomendacion_plantilla(resultado_plantilla)


## 💬 Bloque 9 — Chatbot con Gradio

Interfaz conversacional que permite hacer preguntas sobre jugadores y la plantilla recomendada.

**Arquitectura — Orquestador con Function Calling:**
El chatbot usa el patrón de orquestador con tools de OpenAI. Ante cada pregunta, el modelo decide automáticamente qué fuente consultar:

- `consultar_rag`: busca insights de analistas sobre un jugador concreto.
- `consultar_plantilla`: devuelve la recomendación de alineación ya generada.

**Flujo por tipo de pregunta:**

| Pregunta | Tools llamadas | Llamadas a OpenAI |
|---|---|---|
| "Hola" | Ninguna | 1 |
| "¿Alineo a Pedri?" | `consultar_rag` | 2 |
| "¿Tiene cabida Raphinha en mi plantilla?" | `consultar_rag` + `consultar_plantilla` | 2 |

**Decisiones técnicas:**
- Function calling nativo de OpenAI: más ligero que LangChain para este caso
- Soporte de múltiples tools por turno: el orquestador puede llamar varias en paralelo
- `share=True` en Gradio: genera enlace público desde Colab sin necesidad de ngrok

**Preguntas de ejemplo para la demo:**
- *"¿Debo alinear a Pedri en la jornada 38?"*
- *"¿Qué dicen los expertos sobre Lamine Yamal?"*
- *"¿Cuál es mi 11 recomendado para la jornada 38?"*
- *"¿Quién está mejor esta jornada, Mbappe o Ferran Torres?"*
- *"¿Es seguro poner a Lewandowski esta jornada?"*

**Mejoras futuras:**
- Migración a LangChain con memoria conversacional y agentes ReAct
- Nuevas tools: agente de estadísticas, agente de noticias y lesiones
- Caché de resultados RAG por jugador en sesión para reducir costes

> Ejecuta este bloque como último paso. Se generará un enlace público de Gradio para acceder al chatbot desde el navegador.

In [ ]:
# --- Tools que el orquestador puede llamar ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "consultar_rag",
            "description": "Consulta los insights de los analistas de YouTube sobre un jugador concreto. Úsala cuando el usuario pregunte por un jugador específico, su forma, si debe alinearlo, etc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "jugador": {
                        "type": "string",
                        "description": "Nombre del jugador sobre el que consultar"
                    }
                },
                "required": ["jugador"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_plantilla",
            "description": "Consulta la recomendación de plantilla ya generada por el agente analista. Úsala cuando el usuario pregunte por su 11 recomendado, su alineación óptima, o quiera saber qué jugadores sentar.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]


def ejecutar_tool(nombre_tool: str, argumentos: dict) -> str:
    """Ejecuta la tool seleccionada por el orquestador y retorna el resultado como string."""
    if nombre_tool == "consultar_rag":
        jugador  = argumentos.get("jugador", "")
        insights = obtener_insights_jugador(jugador)

        if not insights:
            return f"No hay insights disponibles para {jugador} en J{JORNADA_OBJETIVO}."

        lineas = [f"Insights de expertos sobre {jugador} (J{JORNADA_OBJETIVO}):"]
        for r in insights:
            m = r["metadata"]
            lineas.append(f"- {m['accion'].upper()} | confianza: {m['confianza']} | {m['canal']}")
            lineas.append(f"  {r['texto'][:200]}")
        return "\n".join(lineas)

    elif nombre_tool == "consultar_plantilla":
        if not resultado_plantilla:
            return "No hay recomendación de plantilla disponible. Ejecuta primero el análisis de plantilla."
        return f"Recomendación de plantilla J{JORNADA_OBJETIVO}:\n{json.dumps(resultado_plantilla, ensure_ascii=False, indent=2)}"

    return "Tool no reconocida."


def orquestador(mensaje: str, historial: list) -> str:
    """
    Orquestador principal con function calling.
    Decide qué tool usar según la pregunta del usuario.
    """
    # Construir mensajes con historial
    mensajes = [
        {
            "role": "system",
            "content": f"""Eres un asistente experto en fantasy football español para LaLiga, Jornada {JORNADA_OBJETIVO}.
Tienes acceso a dos fuentes de información:
1. consultar_rag: insights de analistas de YouTube sobre jugadores concretos
2. consultar_plantilla: recomendación de alineación ya generada para el usuario

Usa siempre las tools disponibles antes de responder.
Responde en español, de forma concisa y útil para el usuario de fantasy."""
        }
    ]

    # Añadir historial
    for human, assistant in historial:
        mensajes.append({"role": "user",      "content": human})
        mensajes.append({"role": "assistant", "content": assistant})

    # Añadir mensaje actual
    mensajes.append({"role": "user", "content": mensaje})

    # Primera llamada — el orquestador decide qué tool usar
    response = client.chat.completions.create(
        model    = "gpt-5.4-mini",
        messages = mensajes,
        tools    = tools,
        tool_choice = "auto"
    )

    mensaje_respuesta = response.choices[0].message

    # Si el modelo quiere usar una tool
    if mensaje_respuesta.tool_calls:
      # Añadir respuesta del modelo al historial
      mensajes.append(mensaje_respuesta)

      # Ejecutar TODAS las tools que el modelo ha pedido
      for tool_call in mensaje_respuesta.tool_calls:
          nombre     = tool_call.function.name
          argumentos = json.loads(tool_call.function.arguments)

          print(f"  🔧 Tool llamada: {nombre} | args: {argumentos}")
          resultado_tool = ejecutar_tool(nombre, argumentos)

          # Añadir resultado de cada tool al historial
          mensajes.append({
              "role":         "tool",
              "tool_call_id": tool_call.id,
              "content":      resultado_tool
          })

      # Segunda llamada con todos los resultados de tools
      response_final = client.chat.completions.create(
          model    = "gpt-4o",
          messages = mensajes,
          tools    = tools,
      )
      return response_final.choices[0].message.content

    # Si responde directamente sin tool
    return mensaje_respuesta.content


# --- Interfaz Gradio ---
with gr.Blocks(title="Fantasy Copilot J38") as demo:
    gr.Markdown("""
    # ⚽ Fantasy Copilot — Jornada 38
    **Pregúntame sobre tu plantilla, jugadores concretos o tu alineación recomendada.**
    """)

    chatbot = gr.Chatbot(
        label  = "Fantasy Copilot",
        height = 500
    )
    msg   = gr.Textbox(
        placeholder = "Escribe tu pregunta sobre fantasy...",
        label       = "Tu pregunta"
    )
    clear = gr.Button("🗑️ Limpiar chat")

    def responder(mensaje, historial):
        historial = historial or []
        print(f"\n💬 Usuario: {mensaje}")
        respuesta = orquestador(mensaje, historial)
        print(f"🤖 Copiloto: {respuesta[:100]}...")
        historial.append((mensaje, respuesta))
        return "", historial

    msg.submit(responder,   [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], None, chatbot)

print("🚀 Lanzando Fantasy Copilot...")
demo.launch(share=True)  # share=True genera enlace público desde Colab